[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Nube1/h-potential-companions/blob/main/methods-series/notebooks/06_hta.ipynb)

# Paper 6: early assessment, value of information, and two outcome measures

A durable therapy against established prophylaxis, judged before the evidence exists. What
is the chance it is cost-effective, what would perfect information be worth, which single
input is worth learning first, and what happens to the decision if you swap the outcome
measure from QALYs to biomedical potential.

## Status, before anything else

- **Every datum here is synthetic.** Nothing in this notebook was measured on a person.
- **Papers 1 to 3 are protocols.** They say what will be done, not what was found. No
  participant has been recruited and no Delphi panel has met.
- **Papers 4 to 6 are method illustrations.** Their inputs show a workflow. They are not
  estimates from any health system and not claims about any product.
- **No figure below is evidence about patients.** Each number is a property of a stated
  model or a stated decision rule, and nothing more.

## Setup

The cell below installs the package straight from GitHub when this runs in Colab, and installs the local checkout when it does not. It prints the resolved version so you know what you are running.

**Runtime.** Reduced run: about 25 seconds. Full run: about 40 seconds.

In [ ]:
# ---------------------------------------------------------------- setup
# Repository holding this package. The repository is private: in Colab, sign
# in to GitHub or clone it yourself before running.
GITHUB_OWNER = "Nube1"
REPO = "h-potential-companions"
SUBDIR = "methods-series"

import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    !pip install -q "hpotential[full] @ git+https://github.com/{GITHUB_OWNER}/{REPO}.git#subdirectory={SUBDIR}"
else:
    # Running from a clone: install the checkout this notebook sits in.
    import importlib.util
    import pathlib
    import subprocess

    if importlib.util.find_spec("hpotential") is None:
        for parent in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
            if (parent / "src" / "hpotential").exists():
                subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-q", "-e", str(parent)], check=True
                )
                break
        else:
            raise RuntimeError(
                "hpotential is not importable and no local checkout was found. "
                "Run this in Colab with access to the repository, or clone it."
            )

import hpotential

print(f"hpotential {hpotential.__version__}")
print(f"running in Colab: {IN_COLAB}")

In [ ]:
# Importing hpotential selects the Agg backend, because the library writes figures to
# files. Put the plots back into the page, then borrow the series figure style.
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from hpotential.utils.plotting import GRID, INK, apply_style, series_color

apply_style()
plt.rcParams["figure.dpi"] = 110  # 200 is for print; this is for a screen

try:
    import ipywidgets as widgets
    from IPython.display import display

    HAVE_WIDGETS = True
except Exception:  # Colab sometimes ships without a working widget stack
    HAVE_WIDGETS = False


def explore(fn, controls, defaults):
    """Draw the static case first, then attach sliders if ipywidgets works.

    The static call means the notebook still teaches something when it is read on
    GitHub, where no widget ever runs.
    """
    fn(**defaults)
    if HAVE_WIDGETS:
        display(widgets.interactive(fn, **controls))
    else:
        print("ipywidgets is unavailable here, so the static case above is all there is.")


print("plots inline, style applied, widgets:", HAVE_WIDGETS)

In [ ]:
# ---------------------------------------------------------------- run size
FULL_RUN = False   # True reproduces the published values. Full run here adds about fifteen seconds.


def banner():
    """Print at the top of every results cell. It has to be impossible to miss."""
    if FULL_RUN:
        print("=" * 74)
        print("FULL RUN: published replicate counts. These are the published values.")
        print("=" * 74)
    else:
        print("=" * 74)
        print("REDUCED RUN: cut replicate counts for speed.")
        print("The numbers below are NOT the published values. Set FULL_RUN = True.")
        print("=" * 74)


banner()

## The model in one paragraph

Six uncertain inputs: remaining years $L_0$, utility gain $q$, waning rate $w$,
implementation access $\alpha$, one-off price, and the annual cost of prophylaxis the
therapy displaces. Per treated person,

$$\Delta\text{QALY} = \alpha \sum_{t} q\,e^{-wt}\,\mathbb{1}\{t < L_0\}\,(1+r)^{-t},
\qquad
\Delta\text{cost} = \text{price} - \text{proph}\sum_{t} e^{-wt}\,\mathbb{1}\{t<L_0\}(1+r)^{-t},$$

and incremental net monetary benefit at threshold $\lambda$ is
$\text{INB} = \lambda\,\Delta\text{QALY} - \Delta\text{cost}$.

Durability drives both sides: the therapy displaces prophylaxis only while its effect
lasts. The utility gain and the durability share a latent quality factor, so a therapy that
helps more also tends to last longer. Sampling them independently would understate the
spread of net benefit and overstate the value of learning either one alone.

In [ ]:
banner()

from hpotential.core.hta import HTAModel, evpi, evppi

model = HTAModel()
print(f"threshold lambda      {model.threshold:,.0f} per QALY")
print(f"eligible population   {model.population} over {model.horizon_years} years")
print(f"discount rate         {model.discount}")
print(f"discounted population multiplier {model.population_multiplier():,.1f}")
print()
for name, spec in model.scenarios.items():
    print(f"{name:<12} waning {spec['waning']}   access {spec['access']}")

In [ ]:
banner()

import time

draws = 100_000 if FULL_RUN else 20_000
start = time.time()
rng = np.random.default_rng(20260815)
results, summaries = {}, {}
for scenario in ("optimistic", "central", "limited"):
    results[scenario] = model.run_scenario(rng, scenario, draws)
    summaries[scenario] = model.summarise(results[scenario], 120)
print(f"{draws} draws per scenario, {time.time() - start:.1f} s")

table = pd.DataFrame({
    s: {"mean incremental QALY": v["mean_incremental_qaly"],
        "mean incremental BMP": v["mean_incremental_bmp"],
        "mean incremental cost": v["mean_incremental_cost"],
        "mean INB": v["mean_INB"],
        "P(cost-effective, QALY)": v["P_cost_effective_QALY"],
        "P(cost-effective, BMP)": v["P_cost_effective_BMP"],
        "P(decisions disagree)": v["P_decision_disagreement"],
        "per-person EVPI": v["per_person_EVPI"],
        "population EVPI": v["population_EVPI"]}
    for s, v in summaries.items()})
table.round(3)

In [ ]:
banner()

fig, axes = plt.subplots(1, 3, figsize=(12.0, 3.6), sharey=True)
for i, scenario in enumerate(("optimistic", "central", "limited")):
    res = results[scenario]
    ax = axes[i]
    take = slice(0, min(6000, draws))
    ax.scatter(res.incremental_qaly[take], res.incremental_cost[take] / 1000, s=3, alpha=0.2,
               color=series_color(i))
    q = np.linspace(0, max(res.incremental_qaly.max(), 0.1), 50)
    ax.plot(q, model.threshold * q / 1000, ls="--", lw=1.1, color=INK)
    ax.axhline(0, lw=0.7, color=GRID)
    ax.set_xlabel("incremental QALYs")
    ax.set_title(f"{scenario}: P(cost-effective) = "
                 f"{summaries[scenario]['P_cost_effective_QALY']:.3f}", fontsize=10)
axes[0].set_ylabel("incremental cost (thousands)")
fig.tight_layout()
plt.show()

print("The dashed line is the threshold. Points below it are cost-effective draws.")
print("Optimistic scenario draws sit low because a durable effect displaces years of")
print("prophylaxis, which is a cost saving, so incremental cost is often negative.")

## What is perfect information worth?

Under uncertainty you take the option with the better expected net benefit and are
sometimes wrong. With perfect information you would be right every time. The gap is

$$\text{EVPI} = E\!\left[\max(\text{INB},0)\right] - \max\!\left(E[\text{INB}],0\right),$$

per person, and it is zero only when every draw has the same sign. Multiply by the
discounted population the decision covers to get what the whole decision is worth.

For one input at a time, expected value of partial perfect information uses the
single-parameter regression method of Strong, Oakley and Brennan: estimate
$g_i(\theta_i) = E[\text{INB} \mid \theta_i]$ by a smoothed conditional mean over
equal-count bins, then $\text{EVPPI}_i = E[\max(g_i,0)] - \max(E[\text{INB}],0)$.

In [ ]:
banner()

evppi_rows = {}
for scenario, summary in summaries.items():
    row = dict(summary["EVPPI"])
    row["(EVPI)"] = summary["per_person_EVPI"]
    evppi_rows[scenario] = row
evppi_table = pd.DataFrame(evppi_rows)
evppi_table.round(1)

In [ ]:
banner()

fig, ax = plt.subplots(figsize=(7.2, 4.0))
inputs = ["L0", "q_gain", "waning", "access", "price", "proph"]
width = 0.26
pos = np.arange(len(inputs))
for i, scenario in enumerate(("optimistic", "central", "limited")):
    shares = [max(summaries[scenario]["EVPPI"][k], 0.0) / summaries[scenario]["per_person_EVPI"]
              for k in inputs]
    ax.bar(pos + (i - 1) * width, shares, width, color=series_color(i), label=scenario)
ax.set_xticks(pos)
ax.set_xticklabels(["remaining years", "utility gain", "waning", "access", "price",
                    "prophylaxis cost"], fontsize=8)
ax.set_ylabel("EVPPI as a share of EVPI")
ax.set_title("Which single input is worth learning first", fontsize=10)
ax.legend(fontsize=8)
plt.show()

central = summaries["central"]
ranked = sorted(central["EVPPI"].items(), key=lambda kv: -kv[1])
print("central scenario, EVPPI per person, largest first:")
for key, value in ranked:
    share = max(value, 0.0) / central["per_person_EVPI"]
    print(f"  {key:<10} {value:10.1f}   {share:6.1%} of EVPI")
print()
print("Shares do not sum to one. Learning two inputs is worth more than the sum of")
print("learning each alone whenever they interact, and a small negative EVPPI is")
print("estimation error in the conditional mean, reported rather than clipped away.")

## Two outcome measures are not one outcome measure

Biomedical potential is $L \times Q$: undiscounted, on a bespoke quality scale, with no
waning term. Three differences from the QALY calculation, and all three push the same way.

In [ ]:
banner()

rows = []
for scenario, summary in summaries.items():
    ratio = summary["mean_incremental_bmp"] / summary["mean_incremental_qaly"]
    rows.append({"scenario": scenario,
                 "incremental QALY": summary["mean_incremental_qaly"],
                 "incremental BMP": summary["mean_incremental_bmp"],
                 "BMP / QALY": ratio,
                 "BMP is larger by": f"{100 * (ratio - 1):.0f}%",
                 "P(cost-effective, QALY)": summary["P_cost_effective_QALY"],
                 "P(cost-effective, BMP)": summary["P_cost_effective_BMP"],
                 "decisions disagree": summary["P_decision_disagreement"]})
pd.DataFrame(rows).round(3)

The central scenario is the one to look at. The incremental benefit measured in biomedical
potential is about 1.88 times the incremental QALY gain, roughly 88 per cent larger, and
the probability of being judged cost-effective moves from 0.410 to 0.469. The two measures
disagree outright on about 6 per cent of draws. A single deterministic score would hide all
of this, which is why the paper does not use one.

## Interactive: move the willingness to pay

The threshold $\lambda$ is a policy choice, not an estimate. Move it and both the
probability of being cost-effective and the value of perfect information move with it. The
EVPI curve peaks where the decision is closest to a coin toss, which is exactly where more
evidence is worth most.

No resampling happens here: the incremental QALYs and costs are fixed, and only
$\text{INB} = \lambda\,\Delta\text{QALY} - \Delta\text{cost}$ is recomputed.

In [ ]:
banner()

lam_grid = np.linspace(5_000, 120_000, 60)
curves = {s: {"p": [], "evpi": []} for s in results}
for scenario, res in results.items():
    for lam in lam_grid:
        inb = lam * res.incremental_qaly - res.incremental_cost
        curves[scenario]["p"].append(float(np.mean(inb > 0)))
        curves[scenario]["evpi"].append(evpi(inb))


def wtp_panel(wtp=30_000):
    fig, axes = plt.subplots(1, 2, figsize=(10.5, 3.6))
    for i, scenario in enumerate(("optimistic", "central", "limited")):
        axes[0].plot(lam_grid / 1000, curves[scenario]["p"], lw=1.5, color=series_color(i),
                     label=scenario)
        axes[1].plot(lam_grid / 1000, np.array(curves[scenario]["evpi"]) / 1000, lw=1.5,
                     color=series_color(i), label=scenario)
    for ax in axes:
        ax.axvline(wtp / 1000, ls="-", lw=1.0, color=series_color(5))
    axes[0].set_ylabel("P(cost-effective)"); axes[0].set_ylim(0, 1)
    axes[1].set_ylabel("per-person EVPI (thousands)")
    for ax in axes:
        ax.set_xlabel("willingness to pay per QALY (thousands)")
    axes[0].legend(fontsize=8)
    fig.tight_layout()
    plt.show()

    print(f"willingness to pay {wtp:,.0f} per QALY")
    for scenario, res in results.items():
        inb = wtp * res.incremental_qaly - res.incremental_cost
        print(f"  {scenario:<12} P(cost-effective) {np.mean(inb > 0):.3f}   "
              f"per-person EVPI {evpi(inb):9,.0f}   "
              f"population EVPI {evpi(inb) * model.population_multiplier():13,.0f}")


explore(wtp_panel, controls=dict(wtp=(5_000, 120_000, 2_500)), defaults=dict(wtp=30_000))

---

## What this does not establish

**Every input is synthetic.** There is no therapy, no trial, no price negotiation and no
patient population behind these numbers. The case is a haemophilia-shaped container for a
workflow.

**EVPI is an upper bound on the value of any study.** No real study delivers perfect
information, so a trial is worth less than the EVPI reported here, sometimes far less.

**EVPPI by single input understates joint value.** The shares do not add up, and learning
two correlated inputs together is worth more than learning each alone. A small negative
value is estimation error in the conditional mean and marks an input carrying essentially
no information value.

**The biomedical-potential comparison is a caution, not a proposal.** The point is that two
measures which both rise with health can order the same decisions differently. It is not a
recommendation to use either one.

**The scenarios are structural assumptions, not probabilities.** Nothing here says the
central scenario is more likely than the limited one.

**A reduced run does not reproduce the published values.** EVPPI is the worst affected,
since each conditional mean is then built from far fewer draws per bin.